# BMC - GBIF

The BMC library offers a simplified API with the GBIF API. This simplified functionality is presented in the `datasource.gbif` module containting the submodules `interface` and `sql`.

1) `interface` provides a simplified way to handle taxonomic matching and abstract a lot of `pygbif` behaviour. The main focus of this submodule is the `fetch_taxon_info` function which can be used to perform name matching between a list of names directly to the GBIF or COL taxonomic identifiers
2) `sql` provides a simplified way of querying GBIF data using the SQL API. The main focus of this submodule is the `generate_sql_query` function that generates SQL ready strings used to query taxonomy, space and time according to their specifications.

Using the `datasource` we implement a standardized processing workflow for the `gbif_cube`. GBIF data generally is queried using either point data or pregridded information according to the grids defined within the SQL API. As such the resulting data that is returned from the database is vector data and as such `gbif_cube` is a child of the `spatiotemporal_vector_cube` class. The implemented workflow processes either the processed gridded data or the raw point data to be aligned with the other cubes produced in the engine

## GBIF-Datasource

## `interface`

### `fetch_taxon_info`

#### Overview

The central function of the `interface` that the user has to handle is the `fetch_taxon_info`. Upon inspecting the function we see that the goal of the function is to perform matching to either the GBIF or COL backbone and give an overview of matching type and the associated key for the queried names back to the user. The end result of the function is 
1) `taxonomic_df`, containing all the matches that have a 1-1 mapping in the taxonomic backbone of interest
2) `mismatch_df`, containing all the matches that can not be resolved exactly. By default this also includes fuzzy, variant and higherrank matches

In [14]:
from bmc.datasource.gbif import interface
interface.fetch_taxon_info?

Signature:
interface.fetch_taxon_info(
    species_names: list[str],
    out_file: str = '',
    out_path: str = '',
    mismatch_file: str = '',
    keep_higherrank: bool = False,
    keep_fuzzy: bool = False,
    max_workers: int = 8,
    col_backbone: bool = False,
    col_uuid: str = '7ddf754f-d193-4cc9-b351-99906754a03b',
) -> tuple[pandas.DataFrame, pandas.DataFrame]
Docstring:
Parse a list of taxonomic names and partition them by query accuracy.

Batch processes name resolutions, splits successful classifications from 
unreliable matches (fuzzy steps, typos, variant names, higher ranks), 
normalizes structural reference keys safely, and optionally records metrics to disk.

Parameters
----------
species_names : list of str
    An array of raw names to validate and ingest.
out_file : str, optional
    The filename target where the validated taxonomic DataFrame is saved as a CSV.
out_path : str, optional
    The directory destination folder path where output CSV files are written.


#### Notes on GBIF matchType

In the Global Biodiversity Information Facility (GBIF) database, the `matchType` field is a classification flag returned by the taxonomic name matching API (such as the Species Match service). It indicates exactly how a user-provided scientific name (`verbatimScientificName`) was resolved against the authoritative GBIF Backbone Taxonomy.

The `matchType` field uses a controlled vocabulary with four possible values:

| `matchType` Value | Description | Common Triggers & Notes |
| :--- | :--- | :--- |
| **`EXACT`** | A perfect string match to a single taxon in the GBIF backbone. | The binomial or polynomial spelling was exactly correct. *Note: This primarily focuses on the name string and may ignore authorship information to make the match.* |
| **`FUZZY`** | A non-exact, probabilistic match. | The API matched the supplied name assuming a slight misspelling, typographical error, or orthographic variant. |
| **`HIGHERRANK`** | Matched to a higher taxonomic grouping (e.g., Genus, Family) rather than the specific rank provided. | 1. The specific species name is missing from the backbone.<br>2. The input intrinsically represents a higher rank.<br>3. **Ambiguity:** Multiple identically spelled names exist (homonyms) and missing authorship forced the system to default to the nearest unambiguous higher rank. |
| **`NONE`** | No match could be made anywhere in the GBIF taxonomic backbone. | Severely malformed names, unmatchable strings (e.g., "Unknown species"), or obscure names completely absent from GBIF's source checklists. |

#### Notes on GBIF `status`

While the `matchType` field describes *how* your input string was matched to the database, the `status` field describes the **taxonomic validity** of the name that was matched. 

When a name is matched, GBIF returns its current standing in the taxonomic backbone. If a name is not the accepted term (e.g., a synonym), the API response will usually provide an `acceptedUsageKey` and `accepted` name string to point you to the correct, currently valid taxon.

The `status` field uses a controlled vocabulary. Below are the primary values you will encounter:

| `status` Value | Description | Notes & Interpretation |
| :--- | :--- | :--- |
| **`ACCEPTED`** | The name is recognized as a currently valid and correct scientific name. | This is the target status for most modern biodiversity data. No further taxonomic resolution is needed. |
| **`SYNONYM`** | The name is recognized as a synonym of another accepted taxon. | General synonym category. GBIF will redirect you to the currently accepted name via the `acceptedUsageKey`. |
| **`DOUBTFUL`** | The validity or application of the name is uncertain or questionable. | Often corresponds to a *nomen dubium*. It represents a name that cannot be assigned to a specific taxon with certainty. |
| **`HETEROTYPIC_SYNONYM`** | A taxonomic (or subjective) synonym. | The synonym and the accepted name are based on **different** type specimens, but taxonomists currently consider them to be the same species. |
| **`HOMOTYPIC_SYNONYM`** | A nomenclatural (or objective) synonym. | The synonym and the accepted name are based on the **exact same** type specimen (e.g., a species was moved to a new genus). |
| **`PROPARTE_SYNONYM`** | A "partial" synonym (*pro parte*). | The original name was applied broadly and has since been split. The synonym applies to only a part of the original taxon's circumscription. |
| **`MISAPPLIED`** | The name was incorrectly applied to a different taxon by an author. | Often found in historical literature where a valid name was mistakenly used to describe a completely different species. |

> **Pro Tip for Data Cleaning:** When resolving taxa, any record returning a `status` other than `ACCEPTED` (or occasionally `DOUBTFUL`) should be updated to use the provided accepted name and accepted taxon key to ensure standardized data alignment.

#### Examples

##### List of exact names

For the list of names that we will use throughout the tutorial we will request information regarding the Asian hornet and some of its established predators. This list has already been vetted and will produce a `taxonomic_df` containing all the names and an empty `mismatch_df` due to feeding the function a well processed list

In [18]:
vespa_velutina_predators = [
    "Pernis apivorus",
    "Dendrocopos major",
    "Parus major",
    "Garrulus glandarius",
    "Pica pica",
    "Merops apiaster",
    "Meles meles",
    "Martes martes",
    "Martes foina",
    "Vespa crabro",
    "Argiope bruennichi",
    "Araneus diadematus"
]
taxonInfo, _ = interface.fetch_taxon_info(["Vespa velutina"]+vespa_velutina_predators)
taxonInfo

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames
0,EXACT,2160133,2160133,"Araneus diadematus Clerck, 1757",SPECIES,ACCEPTED,Araneus diadematus
1,EXACT,2480420,2480420,"Pernis apivorus (Linnaeus, 1758)",SPECIES,ACCEPTED,Pernis apivorus
2,EXACT,5218878,5218878,"Martes martes (Linnaeus, 1758)",SPECIES,ACCEPTED,Martes martes
3,EXACT,1311527,1311527,Vespa crabro Linnaeus,SPECIES,ACCEPTED,Vespa crabro
4,EXACT,5229490,5229490,"Pica pica (Linnaeus, 1758)",SPECIES,ACCEPTED,Pica pica
5,EXACT,5218887,5218887,"Martes foina (Erxleben, 1777)",SPECIES,ACCEPTED,Martes foina
6,EXACT,9705453,9705453,"Parus major Linnaeus, 1758",SPECIES,ACCEPTED,Parus major
7,EXACT,2475443,2475443,"Merops apiaster Linnaeus, 1758",SPECIES,ACCEPTED,Merops apiaster
8,EXACT,2433875,2433875,"Meles meles (Linnaeus, 1758)",SPECIES,ACCEPTED,Meles meles
9,EXACT,2477968,2477968,"Dendrocopos major (Linnaeus, 1758)",SPECIES,ACCEPTED,Dendrocopos major


As is clear in the table, all `matchType` entries are `EXACT` and the `status` entries are `ACCEPTED` and as such this represents a correct and complete list. The `mismatch_df` returned is an empty df

In [16]:
_

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames


The dataframe was above was generated against the GBIF taxonomic backbone. Optionally we can also generate the equivalent table using the COL identfiers by including the optional argument `col_backbone=True`

In [19]:
taxonInfo, _ = interface.fetch_taxon_info(["Vespa velutina"]+vespa_velutina_predators, col_backbone=True)
taxonInfo

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames
0,EXACT,G59D,G59D,"Araneus diadematus Clerck, 1757",SPECIES,ACCEPTED,Araneus diadematus
1,EXACT,4F6YZ,4F6YZ,"Pernis apivorus (Linnaeus, 1758)",SPECIES,ACCEPTED,Pernis apivorus
2,EXACT,3Y9W2,3Y9W2,"Martes martes (Linnaeus, 1758)",SPECIES,ACCEPTED,Martes martes
3,EXACT,5B9T3,5B9T3,Vespa crabro Linnaeus,SPECIES,ACCEPTED,Vespa crabro
4,EXACT,4HPXM,4HPXM,"Pica pica (Linnaeus, 1758)",SPECIES,ACCEPTED,Pica pica
5,EXACT,3Y9VW,3Y9VW,"Martes foina (Erxleben, 1777)",SPECIES,ACCEPTED,Martes foina
6,EXACT,75SVV,75SVV,"Parus major Linnaeus, 1758",SPECIES,ACCEPTED,Parus major
7,EXACT,3ZW55,3ZW55,"Merops apiaster Linnaeus, 1758",SPECIES,ACCEPTED,Merops apiaster
8,EXACT,3ZDRX,3ZDRX,"Meles meles (Linnaeus, 1758)",SPECIES,ACCEPTED,Meles meles
9,EXACT,6CJDB,6CJDB,"Dendrocopos major (Linnaeus, 1758)",SPECIES,ACCEPTED,Dendrocopos major


##### List containing errors

We will now illustrate the result of a list containing names that do not resolve properly in the API. We will include a mix of different types of wrong matches and how to deal with them appropriately. The list wil be as follows


| Input String (`verbatimScientificName`) | `matchType` | `status` | Resolved GBIF Name | Explanation |
| :--- | :--- | :--- | :--- | :--- |
| **`Vulpes vulpes`** | `EXACT` | `ACCEPTED` | *Vulpes vulpes* | The perfect, accepted spelling of the Red Fox. It matches directly to a valid species. |
| **`Felis concolor`** | `EXACT` | `HOMOTYPIC_SYNONYM` | *Puma concolor* | Perfectly spelled, but taxonomically outdated. It is an exact string match to a known synonym, which points to the accepted name *Puma concolor*. |
| **`Pantera leo`** | `FUZZY` | `ACCEPTED` | *Panthera leo* | The genus is missing an 'h'. The API catches the typo, makes a fuzzy match, and successfully lands on the accepted lion species. |
| **`Fellis concolor`** | `FUZZY` | `HOMOTYPIC_SYNONYM` | *Puma concolor* | A double-layer correction. The API fixes the typo ("Fellis" to "Felis") via a fuzzy match, realizes *Felis concolor* is a synonym, and points to *Puma concolor*. |
| **`Glocianus punctiger`** | `HIGHERRANK` | `ACCEPTED` | *Glocianus* (Genus) | **Homonym conflict.** Two different authors described a beetle named *Glocianus punctiger*. Because no authorship was provided in the input, the API cannot safely guess which one is meant, so it defaults to matching at the Genus level. |
| **`Canis crypticus`** | `HIGHERRANK` | `ACCEPTED` | *Canis* (Genus) | A fictional or completely undocumented species name. The API recognizes the valid genus *Canis*, but drops the unrecognized specific epithet. |
| **`Eohippus crypticus`** | `HIGHERRANK` | `SYNONYM` | *Hyracotherium* (Genus) | Fictional species placed in an invalid genus. The API matches the genus *Eohippus*, recognizes that genus is a synonym of *Hyracotherium*, and defaults the match to that accepted genus. |
| **`Trachodon mirabilis`** | `EXACT` | `DOUBTFUL` | *Trachodon mirabilis* | Perfectly spelled, but it is a *nomen dubium* (based on fossil teeth that cannot be confidently assigned to a specific modern dinosaur lineage). The name exists, but its application is uncertain. |
| **`Unknown insect sp.`** | `NONE` | *N/A* | *None* | A completely unmatchable string with no recognizable taxonomic structure in the GBIF backbone. |

In [24]:
gbif_test_names = [
    "Vulpes vulpes",
    "Felis concolor",
    "Pantera leo",
    "Fellis concolor",
    "Glocianus punctiger",
    "Canis crypticus",
    "Eohippus crypticus",
    "Trachodon mirabilis",
    "Unknown insect sp."
]

taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names)
taxonInfo_errors

Mismatches encountered of type {'HIGHERRANK', 'VARIANT', 'EXACT', 'NONE'}
Count HIGHERRANK: 3
Count VARIANT: 2
Count EXACT: 3
Count NONE: 1


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5219243.0,5219243,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
5,EXACT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
7,EXACT,8573909.0,8573909,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN


In [25]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
1,VARIANT,5219404.0,5219404.0,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,HIGHERRANK,5219142.0,5219142.0,"Canis Linnaeus, 1758",GENUS,ACCEPTED,Canis crypticus,NaN
3,VARIANT,2435104.0,2435099.0,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No GBIF match
6,HIGHERRANK,4239.0,4239.0,Curculionidae,FAMILY,ACCEPTED,Glocianus punctiger,NaN
8,HIGHERRANK,4830484.0,4830484.0,"Eohippus Marsh, 1876",GENUS,ACCEPTED,Eohippus crypticus,NaN


Based on the mismatches and the `note` and `matchType` the user can manually review how to proceed. If they wish to include `FUZZY` and `HIGHERRANK` matches into their they can toggle this in the function argument by using `keep_fuzzy` and `keep_higherrank` 

In [26]:
taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names, keep_fuzzy=True, keep_higherrank=True)
taxonInfo_errors

Mismatches encountered of type {'HIGHERRANK', 'VARIANT', 'EXACT', 'NONE'}
Count HIGHERRANK: 3
Count VARIANT: 2
Count EXACT: 3
Count NONE: 1


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5219243.0,5219243,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
1,VARIANT,5219404.0,5219404,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,HIGHERRANK,5219142.0,5219142,"Canis Linnaeus, 1758",GENUS,ACCEPTED,Canis crypticus,NaN
3,VARIANT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
5,EXACT,2435104.0,2435099,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
6,HIGHERRANK,4239.0,4239,Curculionidae,FAMILY,ACCEPTED,Glocianus punctiger,NaN
7,EXACT,8573909.0,8573909,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN
8,HIGHERRANK,4830484.0,4830484,"Eohippus Marsh, 1876",GENUS,ACCEPTED,Eohippus crypticus,NaN


In [27]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No GBIF match


Note, matching in one backbone does not ensure that matching will be exactly the same in the other backbone. Depending on the status of the taxonomic backbone or to which extent it is harmonized, results might be different

In [28]:
taxonInfo_errors, mismatch_df = interface.fetch_taxon_info(gbif_test_names, col_backbone=True)
taxonInfo_errors

Mismatches encountered of type {'NONE', 'EXACT', 'FUZZY'}
Count NONE: 3
Count EXACT: 4
Count FUZZY: 2


,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
0,EXACT,5BSG3,5BSG3,"Vulpes vulpes (Linnaeus, 1758)",SPECIES,ACCEPTED,Vulpes vulpes,NaN
5,EXACT,3DXV5,299463461,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Felis concolor,NaN
6,EXACT,NWPFG,NWPFG,"Glocianus punctiger (C.R. Sahlberg, 1835)",SPECIES,ACCEPTED,Glocianus punctiger,NaN
7,EXACT,TG8W8,TG8W8,"Trachodon mirabilis Leidy, 1856",SPECIES,ACCEPTED,Trachodon mirabilis,NaN


In [29]:
mismatch_df

,matchType,usageKey,acceptedUsageKey,scientificName,rank,status,lookupNames,note
1,FUZZY,4CGXP,4CGXP,"Panthera leo (Linnaeus, 1758)",SPECIES,ACCEPTED,Pantera leo,NaN
2,NONE,NaN,NaN,NaN,NaN,NaN,Canis crypticus,No CoL match
3,FUZZY,3DXV5,299463461,"Felis concolor Linnaeus, 1771",SPECIES,SYNONYM,Fellis concolor,NaN
4,NONE,NaN,NaN,NaN,NaN,NaN,Unknown insect sp.,No CoL match
8,NONE,NaN,NaN,NaN,NaN,NaN,Eohippus crypticus,No CoL match


We note that `Canis crypticus` and `Eohippus crypticus` have no existing COL match and thus an alternative name must be found for it

## `sql`

In [10]:
taxonInfo["acceptedUsageKey"].tolist()

['G59D',
 '4F6YZ',
 '3Y9W2',
 '5B9T3',
 '4HPXM',
 '3Y9VW',
 '75SVV',
 '3ZW55',
 '3ZDRX',
 '6CJDB',
 'GGZL',
 '6JYHZ',
 '7G3C6']

## `gbif_cube`